## Setup

In [1]:
master = "spark://spark-master:7077"  # use "spark://spark-master:7077" when running inside the Docker cluster
dataFolder = "/data/project/" if master.startswith("spark://") else "../../compar-lab02-spark-dataframes/data/"


In [2]:
from pyspark.sql import SparkSession

spark_builder = (SparkSession.builder
         .master(master)
         .appName("project")
         .config("spark.ui.showConsoleProgress", "false"))

spark = spark_builder.getOrCreate()

print("Spark UI:", spark.sparkContext.uiWebUrl)
print("Default parallelism (cores seen by the driver):", spark.sparkContext.defaultParallelism)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/15 10:35:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark UI: http://219491c38ef8:4040
Default parallelism (cores seen by the driver): 6


In [ ]:
import sys
sys.path.append("/workspace/project")
from spark_profiler import SparkProfiler

profiler = SparkProfiler(spark)


In [4]:
with profiler.profile("0-smoke-test"):
   
    taxi_df = spark.read.parquet(dataFolder + "yellow_tripdata_2026-01.parquet")
    
    print("Partitions right after read:", taxi_df.rdd.getNumPartitions())
    print("Total columns:", len(taxi_df.columns))
    
    
    print(taxi_df.groupBy("PULocationID").count().orderBy("PULocationID").limit(10).collect())

Partitions right after read: 6
Total columns: 20
[Row(PULocationID=1, count=486), Row(PULocationID=2, count=3), Row(PULocationID=3, count=491), Row(PULocationID=4, count=16613), Row(PULocationID=5, count=4), Row(PULocationID=6, count=95), Row(PULocationID=7, count=10044), Row(PULocationID=8, count=56), Row(PULocationID=9, count=198), Row(PULocationID=10, count=2050)]
[0-smoke-test] wall time: 20.71s (4 stages, 14 tasks) - see profiler.report() / profiler.compare(...)


## Baseline

# M1

In [8]:
import glob

parquet_files = glob.glob(dataFolder + "*.parquet")

raw_trips = spark.read.parquet(*parquet_files)

#checking the schema
raw_trips.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [9]:
import pyspark.sql.functions as sf

# enforcing schema
trips = (raw_trips
    # .withColumn("tpep_pickup_datetime", sf.col("tpep_pickup_datetime").cast("timestamp")) # (col creat/overwr , opp)
    # .withColumn("tpep_dropoff_datetime", sf.col("tpep_dropoff_datetime").cast("timestamp"))
    # .withColumn("PULocationID", sf.col("PULocationID").cast("integer"))
    # .withColumn("DOLocationID", sf.col("DOLocationID").cast("integer"))
    # .withColumn("passenger_count", sf.col("passenger_count").cast("integer"))
    # .withColumn("trip_distance", sf.col("trip_distance").cast("double"))
    # .withColumn("fare_amount", sf.col("fare_amount").cast("double"))
    .withColumn("year", sf.year(sf.col("tpep_pickup_datetime")))
)

# handling the broken data - vars are just holding condition
invalid_fare      = (sf.col("fare_amount").isNull()) | (sf.col("fare_amount") <= 0)
invalid_distance  = (sf.col("trip_distance").isNull()) | (sf.col("trip_distance") <= 0)
invalid_passengers= (sf.col("passenger_count").isNull()) | (sf.col("passenger_count") == 0) | (sf.col("passenger_count") > 6)
invalid_timestamps= (sf.col("tpep_pickup_datetime").isNull()) | (sf.col("tpep_dropoff_datetime").isNull()) | (sf.col("tpep_dropoff_datetime") <= sf.col("tpep_pickup_datetime"))
invalid_locations = (sf.col("PULocationID").isNull()) | (sf.col("DOLocationID").isNull()) | \
                    (~sf.col("PULocationID").between(1, 263)) | (~sf.col("DOLocationID").between(1, 263))

# combining conditions to one cond
invalid_any = invalid_fare | invalid_distance | invalid_passengers | invalid_timestamps | invalid_locations

# 4. Compute and report per-year cleaning statistics (count of records removed per criterion)
# We cast boolean flags to integers (True -> 1, False -> 0) so sum() acts as a counter
cleaning_stats = (trips
    .groupBy("year")
    .agg(
        sf.count("*").alias("total_records_processed"),
        sf.sum(invalid_fare.cast("integer")).alias("removed_invalid_fare"), #for each year applies the statement
        #and each row assigned with boolean, but then we cast integer, and we sum,
        # so it creates new dataframe with year and number
        sf.sum(invalid_distance.cast("integer")).alias("removed_invalid_distance"),
        sf.sum(invalid_passengers.cast("integer")).alias("removed_invalid_passengers"),
        sf.sum(invalid_timestamps.cast("integer")).alias("removed_invalid_timestamps"),
        sf.sum(invalid_locations.cast("integer")).alias("removed_invalid_locations"),
        sf.sum(invalid_any.cast("integer")).alias("total_records_removed")
    )
    .orderBy("year")
)

# we filter from main trips dataset
m1_clean_trips = trips.filter(~invalid_any)

print("--- M1 Data Cleaning Statistics Report ---")
cleaning_stats.show(truncate=False)

--- M1 Data Cleaning Statistics Report ---
+----+-----------------------+--------------------+------------------------+--------------------------+--------------------------+-------------------------+---------------------+
|year|total_records_processed|removed_invalid_fare|removed_invalid_distance|removed_invalid_passengers|removed_invalid_timestamps|removed_invalid_locations|total_records_removed|
+----+-----------------------+--------------------+------------------------+--------------------------+--------------------------+-------------------------+---------------------+
|2008|1                      |0                   |0                       |0                         |0                         |0                        |0                    |
|2009|1                      |0                   |0                       |0                         |0                         |0                        |0                    |
|2025|6                      |1                   |0          

In [12]:
m1_clean_trips.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- year: integer (nullable = true)



# M2

In [10]:
import os

zone_lookup_path = dataFolder + "taxi_zone_lookup.csv"

#loading just 2 cols
zone_lookup = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(zone_lookup_path)
    .select(
        sf.col("LocationID").cast("integer").alias("zone_id"),
        sf.col("Borough").alias("pickup_borough")
    )
)

m2_prepared_trips = (m1_clean_trips
    .withColumn("day_of_week", sf.dayofweek(sf.col("tpep_pickup_datetime")))  
    .withColumn("hour_of_day", sf.hour(sf.col("tpep_pickup_datetime")))
    .withColumn("duration_minutes", 
        (sf.unix_timestamp(sf.col("tpep_dropoff_datetime")) - sf.unix_timestamp(sf.col("tpep_pickup_datetime"))) / 60.0
    )
)

#for optimization performing the broadcast join - avoiding shuffle and makes sense because csv small
#
m2_joined = m2_prepared_trips.join(
    zone_lookup,
    m2_prepared_trips.PULocationID == zone_lookup.zone_id,
    how="inner"
)


m2_demand_analysis = (m2_joined
    .groupBy("pickup_borough", "day_of_week", "hour_of_day")
    .agg(
        sf.count("*").alias("total_trip_count"),
        sf.mean("fare_amount").alias("mean_fare_amount"),
        sf.mean("duration_minutes").alias("mean_trip_duration_min"),
        sf.mean("trip_distance").alias("mean_trip_distance")
    )
    .orderBy("pickup_borough", "day_of_week", "hour_of_day")
)

output_path_m2 = "/tmp/m2_temporal_demand_output"

spark.sparkContext.setJobGroup("M2_ANALYSIS", "M2: Temporal Demand Parquet Write") #for spark ui

(m2_demand_analysis
    .write
    .mode("overwrite")
    .partitionBy("pickup_borough")
    .parquet(output_path_m2)
)

spark.sparkContext.setJobGroup("M2_ANALYSIS", "show 10 lines") 

print("--- M2 Temporal Demand Analysis 10 first ---")
m2_demand_analysis.show(10, truncate=False)

spark.sparkContext.setJobGroup(" ", " ") # cleaning labeling for sparkui

--- M2 Temporal Demand Analysis 10 first ---
+--------------+-----------+-----------+----------------+------------------+----------------------+------------------+
|pickup_borough|day_of_week|hour_of_day|total_trip_count|mean_fare_amount  |mean_trip_duration_min|mean_trip_distance|
+--------------+-----------+-----------+----------------+------------------+----------------------+------------------+
|Bronx         |1          |0          |41              |33.02682926829268 |27.151626016260163    |8.88878048780488  |
|Bronx         |1          |1          |23              |23.434782608695652|20.66521739130435     |5.460434782608696 |
|Bronx         |1          |2          |14              |30.585714285714285|15.167857142857143    |5.3792857142857144|
|Bronx         |1          |3          |13              |36.13846153846154 |23.19871794871795     |9.263846153846155 |
|Bronx         |1          |4          |29              |37.06896551724138 |33.782758620689656    |9.28448275862069  |
|Br

In [11]:
from pyspark.sql.window import Window

spark.sparkContext.setJobGroup("M3_STATS", "M3: Zone-Level Window Statistics Calculation")

m3_prepared = (m1_clean_trips
    .withColumn("year_month", sf.date_format(sf.col("tpep_pickup_datetime"), "yyyy-MM"))
    .join(
        zone_lookup,
        m1_clean_trips.PULocationID == zone_lookup.zone_id,
        how="inner"
    )
    .filter(sf.col("pickup_borough").isNotNull())
)


m3_monthly_zone_stats = (m3_prepared
    .groupBy("pickup_borough", "PULocationID", "year_month") #we need pickup borough for laters window
    .agg(
        sf.count("*").alias("monthly_trip_count"),
        sf.mean("fare_amount").alias("mean_fare_amount"),
        sf.stddev("fare_amount").alias("std_fare_amount"),
        sf.mean("trip_distance").alias("mean_trip_distance"),
        sf.stddev("trip_distance").alias("std_trip_distance")
    )
)

# it assigns to the buckets, we dont collapse rows here like in a group by
w_borough_monthly_rank = Window.partitionBy("pickup_borough", "year_month").orderBy(sf.col("monthly_trip_count").desc())

w_borough_monthly_total = Window.partitionBy("pickup_borough", "year_month")

w_zone_rolling_3m = (Window
    .partitionBy("PULocationID")
    .orderBy("year_month")
    .rowsBetween(-2, 0)
)


m3_final = (m3_monthly_zone_stats
    
    .withColumn("zone_volume_rank", sf.rank().over(w_borough_monthly_rank)) #here the index is created
    
    # for each neigbourhood and month, duplicated the same value of total
    .withColumn("borough_monthly_trips", sf.sum("monthly_trip_count").over(w_borough_monthly_total)) 

    # each id's total count over the neighberhoods total
    .withColumn("zone_monthly_trip_share", sf.col("monthly_trip_count") / sf.col("borough_monthly_trips"))
    
    .withColumn("rolling_3m_trip_share", sf.sum("zone_monthly_trip_share").over(w_zone_rolling_3m))
    
    # avg and std for rolling 3 months
    .withColumn("rolling_volume_mean", sf.avg("monthly_trip_count").over(w_zone_rolling_3m))
    .withColumn("rolling_volume_std", sf.stddev("monthly_trip_count").over(w_zone_rolling_3m))
    
    # Flag zones where |current volume - rolling mean| > 1.5 * rolling std
    .withColumn(
        "is_volume_anomaly",
        sf.when(
            sf.col("rolling_volume_std").isNotNull() & (sf.col("rolling_volume_std") > 0),
            sf.abs(sf.col("monthly_trip_count") - sf.col("rolling_volume_mean")) > (1.5 * sf.col("rolling_volume_std"))
        ).otherwise(False)
    )
)

# -------------------------------------------------------------------------
# Step 5: Export to /tmp output path
# -------------------------------------------------------------------------
output_path_m3 = "/tmp/m3_zone_window_stats_output"

(m3_final
    .write
    .mode("overwrite")
    .partitionBy("pickup_borough")
    .parquet(output_path_m3)
)

print("--- M3 Zone-Level Window Statistics (First 10 Rows) ---")
m3_final.show(10, truncate=False)

# Reset job group
spark.sparkContext.setJobGroup("", "")

--- M3 Zone-Level Window Statistics (First 10 Rows) ---
+--------------+------------+----------+------------------+------------------+------------------+------------------+------------------+----------------+---------------------+-----------------------+---------------------+-------------------+------------------+-----------------+
|pickup_borough|PULocationID|year_month|monthly_trip_count|mean_fare_amount  |std_fare_amount   |mean_trip_distance|std_trip_distance |zone_volume_rank|borough_monthly_trips|zone_monthly_trip_share|rolling_3m_trip_share|rolling_volume_mean|rolling_volume_std|is_volume_anomaly|
+--------------+------------+----------+------------------+------------------+------------------+------------------+------------------+----------------+---------------------+-----------------------+---------------------+-------------------+------------------+-----------------+
|EWR           |1           |2026-01   |110               |93.49636363636364 |42.787005395239376|1.59727272727